In [1]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

/Users/tsiameh/Desktop/PythonCourse/ai-ml-course/lesson20/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.2.2
Transformers: 4.57.6
CUDA available: False


In [2]:
from huggingface_hub import login
import os

In [3]:
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Download an entire model repository

In [4]:
from huggingface_hub import snapshot_download

In [5]:
model_path = snapshot_download(repo_id="Qwen/Qwen3-0.6B", local_dir="./models/qwen3-0.6B")

print(f"Model downloaded to: {model_path}")

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 94.42it/s]

Model downloaded to: /Users/tsiameh/Desktop/PythonCourse/ai-ml-course/lesson20/models/qwen3-0.6B


## Download a specific file

In [6]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="Qwen/Qwen3-0.6B",
    filename="vocab.json", # just this one line
    local_dir="./qwen3-vocab"
)

print(file_path)

qwen3-vocab/vocab.json


## Download and Load a model using Transformers


In [7]:
from transformers import AutoTokenizer, AutoModel

In [8]:
model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModel.from_pretrained(model_name)

print("Model loaded successfully!")

Model loaded successfully!


In [9]:
text = "Model loaded successfully!"

token_ids = tokenizer.encode(text)
print(token_ids)

[1712, 6661, 7790, 0]


In [10]:
tokenizer.decode(token_ids)

'Model loaded successfully!'

### see the actual token ids

In [11]:
text = "Hello, how are you?"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

Tokens:
['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']

Token IDs:
[9707, 11, 1246, 525, 498, 30]


In [12]:
# Convert text to token IDs
inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[9707,   11, 1246,  525,  498,   30]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [13]:
input_ids = inputs["input_ids"]

print(input_ids)

tensor([[9707,   11, 1246,  525,  498,   30]])


### feed the token ids into the model

In [14]:
outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

print(outputs)

BaseModelOutputWithPast(last_hidden_state=tensor([[[ 7.0959e+00,  2.7858e+01, -1.4415e-01,  ..., -1.2679e+00,
           6.1285e-01,  1.1405e+00],
         [ 3.0073e-02, -1.2989e+00, -1.3900e+00,  ...,  3.0444e-02,
          -2.8465e+00, -2.2351e-01],
         [-1.7102e+00, -5.4694e+00, -1.4694e+00,  ..., -7.3180e-01,
           3.3693e+00,  2.4752e+00],
         [-7.1531e-01, -7.0283e+00, -1.5931e+00,  ..., -1.2184e+00,
           2.1747e-01, -9.7177e-01],
         [-8.3713e-01, -4.9342e+00, -1.2220e+00,  ..., -4.6809e+00,
           1.1284e+00, -8.6650e-01],
         [ 1.4828e-01,  3.1367e+01, -1.3529e+00,  ...,  1.5639e+00,
           6.3422e-01, -1.7506e-02]]], grad_fn=<MulBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer

## Qwen3 0.6B is a causal language model, 
- so it obtains logits for the next token prediction

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

### Convert text to token IDs

In [16]:
text = "The capital of France is"

inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[ 785, 6722,  315, 9625,  374]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}


In [17]:
input_ids = inputs["input_ids"]

print(input_ids)
print(input_ids.shape)

tensor([[ 785, 6722,  315, 9625,  374]])
torch.Size([1, 5])


## Flow

"The capital of France is"
            ↓
        Tokenizer
            ↓
       input_ids
            ↓
         Qwen3
            ↓
         logits
            ↓
         argmax
            ↓
      next token ID
            ↓
         Decode

### Feed the token IDs into the Qwen3

In [18]:
outputs = model(
    input_ids=input_ids,
    attention_mask=inputs["attention_mask"]
)

print(outputs.logits.shape)

# torch.Size([1, sequence_length, vocab_size])

torch.Size([1, 5, 151936])


In [19]:
print(outputs.logits)

tensor([[[ 4.8132,  4.9894,  3.0482,  ...,  1.0447,  1.0447,  1.0447],
         [ 5.3644,  8.1180,  1.1067,  ..., -2.9590, -2.9590, -2.9590],
         [ 3.7584,  5.5630,  0.3325,  ..., -2.9157, -2.9157, -2.9157],
         [ 9.7910,  7.9012,  3.7287,  ..., -2.8894, -2.8894, -2.8894],
         [ 5.1312,  8.2129,  3.1985,  ..., -3.3041, -3.3041, -3.3041]]],
       grad_fn=<UnsafeViewBackward0>)


In [21]:
outputs.logits[:, -1, :]

tensor([[ 5.1312,  8.2129,  3.1985,  ..., -3.3041, -3.3041, -3.3041]],
       grad_fn=<SliceBackward0>)

In [22]:
# the last position is the interesting one for generation 

next_token_logits = outputs.logits[:, -1, :]

print(next_token_logits)

# torch.Size([1, vocab_size])

tensor([[ 5.1312,  8.2129,  3.1985,  ..., -3.3041, -3.3041, -3.3041]],
       grad_fn=<SliceBackward0>)


### Argmax

Qwen3 predicts scores:

Paris       → 12.8
London      →  4.2
Berlin      →  1.7
Rome        →  0.9
Tokyo       → -2.1

                 ↓

torch.argmax()

                 ↓

Paris has the largest score: 12.8

                 ↓

Token ID for "Paris"

                 ↓

tokenizer.decode(token_id)

                 ↓

"Paris"

In [23]:
scores = torch.tensor([
    1.2,
    5.7,
    2.3,
    9.8,
    4.1
])

print(scores)

index = torch.argmax(scores)

print(index)

tensor([1.2000, 5.7000, 2.3000, 9.8000, 4.1000])
tensor(3)


### Get the most likely next token

In [26]:
next_token_id = outputs.logits[:, -1, :].argmax(dim=-1)

print(next_token_id)

tensor([12095])


In [28]:
# convert that ID back into text
next_token = tokenizer.decode(next_token_id)

print(next_token)

 Paris


### See the tokens and IDs

In [29]:
text = "The capital of France is Paris"

inputs = tokenizer(text, return_tensors="pt")

input_ids = inputs["input_ids"][0]

tokens = tokenizer.convert_ids_to_tokens(input_ids)

for token_id, token in zip(input_ids, tokens):
    print(token_id.item(), "→", repr(token))

785 → 'The'
6722 → 'Ġcapital'
315 → 'Ġof'
9625 → 'ĠFrance'
374 → 'Ġis'
12095 → 'ĠParis'


### Text Generation

In [26]:
text = "The capital of Ghana is"

inputs = tokenizer(text,return_tensors="pt")

print(inputs)

outputs = model.generate(**inputs, max_new_tokens=20)

print(outputs[0])

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

{'input_ids': tensor([[  785,  6722,   315, 47568,   374]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}
tensor([  785,  6722,   315, 47568,   374,   279,  6722,   315,   279,  3146,
           11,   323,   432,   374,   279,  6722,   315,   279,  3146,   594,
         8584,    13,   576,  6722,   315])
The capital of Ghana is the capital of the country, and it is the capital of the country's economy. The capital of


### Complete Demo

In [27]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Load Qwen3
model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

# 2. Input text
text = "The capital of France is"

# 3. Convert text to token IDs
inputs = tokenizer(text,return_tensors="pt")

print("Input IDs:")
print(inputs["input_ids"])

print("\nInput Tokens:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

# 4. Feed token IDs into the model
with torch.no_grad():
    outputs = model(**inputs)

# 5. Get logits for the LAST position
next_token_logits = outputs.logits[:, -1, :]

print("\nLogits shape:")
print(next_token_logits.shape)

# 6. Find token ID with the highest score
next_token_id = torch.argmax(next_token_logits, dim=-1)

# 7. Convert token ID back to text
next_token = tokenizer.decode(next_token_id)

print("\nPredicted Token ID:", next_token_id.item())
print("Predicted Token:", repr(next_token))

Input IDs:
tensor([[ 785, 6722,  315, 9625,  374]])

Input Tokens:
['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']

Logits shape:
torch.Size([1, 151936])

Predicted Token ID: 12095
Predicted Token: ' Paris'


## Text-Generation

In [28]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

result = generator("Artificial Intelligence is", max_new_tokens=50)

print(result[0]["generated_text"])

Device set to use mps:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/Users/tsiameh/Desktop/PythonCourse/ai-ml-course/lesson20/.venv/lib/python3.12/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Artificial Intelligence is in the process of being developed by many companies around the world, including Google. The company's AI and machine learning systems are being used to help shape the world's laws, and it's unclear whether that change will have long-term effects.



In [29]:
def generate_text(model, tokenizer, prompt, max_new_tokens=20):
    """
    Generate text one token at a time using greedy decoding (argmax).
    """

    # Put model in evaluation mode
    model.eval()

    # Convert prompt to token IDs
    inputs = tokenizer(prompt,return_tensors="pt")

    # Start with the prompt's token IDs
    input_ids = inputs["input_ids"]

    # Generate tokens one at a time
    with torch.no_grad():

        for _ in range(max_new_tokens):

            # Feed token IDs into the model
            outputs = model(input_ids=input_ids)

            # Get logits for the last position
            next_token_logits = outputs.logits[:, -1, :]

            # Choose the token with the highest score
            next_token_id = torch.argmax(next_token_logits,dim=-1,keepdim=True)

            # Add the new token to the sequence
            input_ids = torch.cat([input_ids, next_token_id],dim=1)

            # Stop if EOS token is generated
            if next_token_id.item() == tokenizer.eos_token_id:
                break

    # Convert all token IDs back to text
    generated_text = tokenizer.decode(input_ids[0],skip_special_tokens=True)

    return generated_text

In [34]:
prompt = "write a poem about france"

result = generate_text(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=2
)

print(result)

write a poem about france's history


In [35]:

def generate_text_verbose(model, tokenizer, prompt, max_new_tokens=20):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"]

    print("Prompt:")
    print(prompt)
    print("\nGenerated tokens:")

    with torch.no_grad():

        for _ in range(max_new_tokens):

            outputs = model(
                input_ids=input_ids
            )

            next_token_logits = outputs.logits[:, -1, :]

            next_token_id = torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True
            )

            # Decode only the newly predicted token
            next_token = tokenizer.decode(
                next_token_id[0],
                skip_special_tokens=False
            )

            print(
                f"Token ID: {next_token_id.item()} "
                f"→ {repr(next_token)}"
            )

            input_ids = torch.cat(
                [input_ids, next_token_id],
                dim=1
            )

            if next_token_id.item() == tokenizer.eos_token_id:
                print("EOS reached.")
                break

    return tokenizer.decode(
        input_ids[0],
        skip_special_tokens=True
    )

In [36]:
prompt = "The capital of France is"

result = generate_text_verbose(
    model,
    tokenizer,
    prompt,
    max_new_tokens=10
)

print("\nFinal text:")
print(result)

Prompt:
The capital of France is

Generated tokens:
Token ID: 12095 → ' Paris'
Token ID: 13 → '.'
Token ID: 576 → ' The'
Token ID: 6722 → ' capital'
Token ID: 315 → ' of'
Token ID: 15344 → ' Italy'
Token ID: 374 → ' is'
Token ID: 21718 → ' Rome'
Token ID: 13 → '.'
Token ID: 576 → ' The'

Final text:
The capital of France is Paris. The capital of Italy is Rome. The


## Save the downloaded model locally

In [37]:
from transformers import AutoTokenizer, AutoModel

model_name = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

('./my_model/tokenizer_config.json',
 './my_model/special_tokens_map.json',
 './my_model/vocab.txt',
 './my_model/added_tokens.json',
 './my_model/tokenizer.json')

## Upload a model to huggingface 

In [38]:
# method 1

model.push_to_hub("worldboss/bert-base-uncased-0.6B")

tokenizer.push_to_hub("worldboss/bert-base-uncased-0.6B")

Processing Files (1 / 1): 100%|██████████|  438MB /  438MB, 22.6MB/s  
New Data Upload: 100%|██████████|  130MB /  130MB, 7.17MB/s  


CommitInfo(commit_url='https://huggingface.co/worldboss/bert-base-uncased-0.6B/commit/2145ccdbad7ab6a29702c212078262da95a07253', commit_message='Upload tokenizer', commit_description='', oid='2145ccdbad7ab6a29702c212078262da95a07253', pr_url=None, repo_url=RepoUrl('https://huggingface.co/worldboss/bert-base-uncased-0.6B', endpoint='https://huggingface.co', repo_type='model', repo_id='worldboss/bert-base-uncased-0.6B'), pr_revision=None, pr_num=None)

In [ ]:
# method 2
from transformers import AutoModel, AutoTokenizer

model_name = "google/gemma-4-31B-it"

model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# repo_name = "my-gemma-4-31B-it-model"

# model.push_to_hub(repo_name)
# tokenizer.push_to_hub(repo_name)

model.push_to_hub("worldboss/my-gemma-4-31B-it-model")
tokenizer.push_to_hub("worldboss/my-gemma-4-31B-it-model")

ValueError: The checkpoint you are trying to load has model type `gemma4` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

## Create the repository first

In [43]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id="worldboss/my-model",
    repo_type="model",
    exist_ok=True
)

RepoUrl('https://huggingface.co/worldboss/my-model', endpoint='https://huggingface.co', repo_type='model', repo_id='worldboss/my-model')

In [44]:
api.upload_folder(
    folder_path="./my_model",
    repo_id="worldboss/my-model",
    repo_type="model"
)

Processing Files (1 / 1): 100%|██████████|  438MB /  438MB, 28.1MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


CommitInfo(commit_url='https://huggingface.co/worldboss/my-model/commit/e8937e51a0350f8a74662b2abe37d507aee04b49', commit_message='Upload folder using huggingface_hub', commit_description='', oid='e8937e51a0350f8a74662b2abe37d507aee04b49', pr_url=None, repo_url=RepoUrl('https://huggingface.co/worldboss/my-model', endpoint='https://huggingface.co', repo_type='model', repo_id='worldboss/my-model'), pr_revision=None, pr_num=None)

## Complete example

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "your-username/my-awesome-model"
LOCAL_MODEL_PATH = "./my_model"

api = HfApi()

# Create repository
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True
)

# Upload model files
api.upload_folder(
    folder_path=LOCAL_MODEL_PATH,
    repo_id=REPO_ID,
    repo_type="model"
)

print(f"Successfully uploaded to: https://huggingface.co/{REPO_ID}")